# 04 · Gradient Boosted Decision Trees

Gradient Boosted Decision Trees (GBDT) build an additive prediction model by fitting
small regression trees sequentially. Under squared-error loss, each new tree learns
the residuals left by the current model:

$$
F_0(x)=\overline y_{\mathrm{train}},\qquad
r_{im}=y_i-F_{m-1}(x_i),
$$

$$
h_m=\arg\min_{h\in\mathcal H}\sum_{i=1}^{n}(r_{im}-h(x_i))^2,
\qquad F_m(x)=F_{m-1}(x)+\eta h_m(x).
$$

The final model is $F_M(x)=F_0(x)+\eta\sum_{m=1}^{M}h_m(x)$, where $M$ is the
number of boosting rounds and $\eta$ is the learning rate. Tree size and leaf
regularization control model complexity. Trees can capture nonlinear relationships
and interactions between predictors.

This notebook uses histogram-based GBDT to predict sector-minus-SPY returns and
selects hyperparameters using time-ordered validation.

## 1. Environment and parameter grid

Search maximum leaves $\{3,5,7\}$, minimum leaf sizes $\{10,20,50\}$ and
L2 leaf regularization $\{0,1,10\}$. This gives 27 configurations per scheme,
or 81 across S0/S1/S2. Learning rate is 0.03, maximum depth is 4 and maximum
histogram bins is 127. Early stopping selects boosting rounds from the pooled
2018-2022 validation loss curve, with a maximum of 50 rounds, patience of
10 rounds and an MSE improvement tolerance of $10^{-8}$.


In [ ]:
from pathlib import Path
from itertools import product
import json
import os
import time

os.environ['OMP_NUM_THREADS'] = '1'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.utils.parallel import Parallel, delayed
from threadpoolctl import threadpool_limits
from IPython.display import display

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "requirements.txt").is_file()
             and (p / "notebooks").is_dir() and (p / "scripts").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project directory or a subdirectory.")
if not (ROOT / "data/processed/panel_spec.json").is_file():
    raise FileNotFoundError("Run notebooks 01 and 02 to completion before notebook 04.")
OUT = ROOT/'results/gbdt'
OUT.mkdir(parents=True, exist_ok=True)
SCHEMES = ['S0', 'S1', 'S2']
SEED = 5205
LEAVES = [3, 5, 7]
MIN_SAMPLES_LEAF = [10, 20, 50]
L2 = [0.0, 1.0, 10.0]
ES_MAX_ROUNDS = 50
ES_PATIENCE = 10
ES_TOL = 1e-8
FIXED = dict(loss='squared_error', learning_rate=0.03, max_depth=4,
            max_bins=127, early_stopping=False,
            max_features=1.0, categorical_features=None, random_state=SEED)
GRID = pd.DataFrame(product(LEAVES, MIN_SAMPLES_LEAF, L2), columns=[
    'max_leaf_nodes', 'min_samples_leaf', 'l2_regularization'])
config = dict(model='HistGradientBoostingRegressor', fixed=FIXED,
            early_stopping=dict(max_rounds=ES_MAX_ROUNDS, patience=ES_PATIENCE,
                                        tolerance=ES_TOL, validation='pooled monthly 2018-2022 MSE',
                                        refit='selected rounds on all eligible training rows'),
            grid=GRID.to_dict('records'), candidates_per_scheme=len(GRID),
            training='monthly expanding from 2008; mature labels only',
            validation='2018-2022', test='2023-2026-09',
            selection='pooled validation MSE per scheme; ties: fewer leaves, fewer rounds, larger L2, larger minimum leaf size',
            provenance='Exploratory grid; previous project test results already known')
(OUT/'model_config.json').write_text(json.dumps(config, indent=2)+'\n', encoding='utf-8')
display(GRID)

## 2. Shared inputs and monthly training samples

Train separate models for nine sectors under S0, S1 and S2. Features are observed
at 18:00 New York time; the target is next-open to following-open sector cash
return minus SPY cash return. Training expands from 2008, using only labels
realized by the previous trading day's 18:00 cutoff. Validation covers 2018–2022
and testing covers 2023–September 2026.

In [ ]:
spec = json.loads((ROOT/'data/processed/panel_spec.json').read_text(encoding='utf-8'))
panel = pd.read_csv(ROOT/spec['shared_panel'], parse_dates=['date', 'entry_date', 'exit_date'],
                    float_precision='round_trip')
for col in ['signal_at', 'label_available_at']:
    panel[col] = pd.to_datetime(panel[col], utc=True).dt.tz_convert('America/New_York')
calendar = pd.DatetimeIndex(sorted(panel.date.unique()))
data = {}
for ticker, files in spec['input_files'].items():
    info = panel.loc[panel.ticker.eq(ticker)].sort_values('date').reset_index(drop=True)
    info['month'] = info.date.dt.strftime('%Y-%m')
    xs = {}
    for scheme in SCHEMES:
        features = pd.read_csv(ROOT/files[scheme], float_precision='round_trip')
        xs[scheme] = features[spec['predictors_by_target'][ticker][scheme]].to_numpy(float)
    data[ticker] = (info, xs)

schedule = {}
for month in pd.period_range('2018-01', '2026-09', freq='M'):
    first = calendar[calendar.to_period('M') == month][0]
    prior = calendar[calendar.get_loc(first)-1]
    schedule[str(month)] = (prior + pd.Timedelta(hours=18)).tz_localize('America/New_York')

In [ ]:
def month_data(month, ticker, scheme):
    info, xs = data[ticker]
    cutoff = schedule[month]
    train = (info.date.ge('2008-01-01') & info.common_features_valid & info.label_valid
            & info.signal_at.le(cutoff) & info.label_available_at.le(cutoff)).to_numpy()
    pred = (info.month.eq(month) & info.common_features_valid).to_numpy()
    y_train = info.loc[train, 'target_relative_return'].to_numpy()
    columns = ['date', 'ticker', 'split', 'common_scoreable', 'entry_date', 'exit_date',
            'target_relative_return', 'sector_forward_cash_return', 'spy_forward_cash_return']
    result = info.loc[pred, columns].copy()
    result['model'], result['scheme'] = 'gbdt', scheme
    result['model_month'], result['fit_cutoff'] = month, cutoff
    result['historical_mean'] = y_train.mean()
    audit = dict(month=month, ticker=ticker, scheme=scheme, fit_cutoff=cutoff,
                training_rows=int(train.sum()),
                latest_label=info.loc[train, 'label_available_at'].max(),
                first_signal=info.loc[pred, 'signal_at'].min())
    return xs[scheme][train], y_train, xs[scheme][pred], result, audit

## 3. Validation hyperparameter search with early stopping

For each leaf count, minimum leaf size and L2 penalty, fit monthly models on all
eligible historical rows and aggregate validation SSE and observation counts at
every boosting round.
Use the existing 2018–2022 validation period without an additional holdout.

Scan each structure's pooled MSE curve in round order. Stop scanning after
`ES_PATIENCE` rounds without an improvement exceeding `ES_TOL` over the last
significant best MSE. Retain the lowest-loss round visited, then select the
structure with the lowest validation MSE. Exact ties prefer fewer leaves,
fewer rounds, larger L2 and larger minimum leaf size.

Monthly paths are computed first; the patience rule selects the stopping point
from these paths. The estimator's internal early stopping is disabled because
round selection uses the shared validation period. Selected parameters are
locked before test evaluation.


In [ ]:
def validation_job(task):
    month, ticker, scheme, leaves, min_leaf, l2 = task
    x_train, y_train, x_pred, frame, _ = month_data(month, ticker, scheme)
    score = frame.common_scoreable.to_numpy()
    y = frame.loc[score, 'target_relative_return'].to_numpy()
    model = HistGradientBoostingRegressor(**FIXED, max_leaf_nodes=leaves,
                                        max_iter=ES_MAX_ROUNDS, min_samples_leaf=min_leaf,
                                        l2_regularization=l2)
    model.fit(x_train, y_train)
    rows = []
    for rounds, prediction in enumerate(model.staged_predict(x_pred[score]), 1):
        rows.append(dict(month=month, scheme=scheme, max_leaf_nodes=leaves,
                        max_iter=rounds, min_samples_leaf=min_leaf, l2_regularization=l2,
                        n=len(y), sse=np.square(y-prediction).sum()))
    return rows

monthly_losses = []
parameter_columns = ['max_leaf_nodes', 'min_samples_leaf', 'max_iter', 'l2_regularization']
started = time.perf_counter()
with threadpool_limits(limits=1), Parallel(n_jobs=4, prefer="threads") as pool:
    for month in [m for m in schedule if m < '2023-01']:
        tasks = [(month, ticker, scheme, leaves, min_leaf, l2)
                 for ticker in sorted(data) for scheme in SCHEMES
                 for leaves, min_leaf, l2 in GRID.itertuples(index=False, name=None)]
        month_rows = [row for rows in pool(delayed(validation_job)(task) for task in tasks) for row in rows]
        monthly_losses.append(pd.DataFrame(month_rows).groupby(
            ['month', 'scheme']+parameter_columns, as_index=False)[['n', 'sse']].sum())
        print('Validation', month, '|', round(time.perf_counter()-started, 1), 's')

validation_losses = pd.concat(monthly_losses, ignore_index=True)
validation_losses.to_csv(OUT/'validation_monthly_losses.csv', index=False)
round_curve = validation_losses.groupby(['scheme']+parameter_columns, as_index=False)[['n', 'sse']].sum()
round_curve['validation_mse'] = round_curve.sse/round_curve.n
round_curve.to_csv(OUT/'validation_round_curve.csv', index=False)
ranking = ['scheme', 'validation_mse', 'max_leaf_nodes', 'max_iter',
           'l2_regularization', 'min_samples_leaf']
order = [True, True, True, True, False, False]
choices = []
for _, curve in round_curve.groupby(
        ['scheme', 'max_leaf_nodes', 'min_samples_leaf', 'l2_regularization']):
    curve = curve.sort_values('max_iter')
    reference_loss, stale_rounds = np.inf, 0
    for row in curve.itertuples():
        if row.validation_mse < reference_loss-ES_TOL:
            reference_loss, stale_rounds = row.validation_mse, 0
        else:
            stale_rounds += 1
        stop_round = row.max_iter
        if stale_rounds >= ES_PATIENCE:
            break
    best = curve.loc[curve.max_iter.le(stop_round)].sort_values(
        ['validation_mse', 'max_iter']).iloc[0].to_dict()
    best['stopping_round'] = stop_round
    choices.append(best)

search = pd.DataFrame(choices).sort_values(ranking, ascending=order)
search.to_csv(OUT/'validation_grid.csv', index=False)
selected = search.groupby('scheme', sort=False).head(1).reset_index(drop=True)
selected.to_csv(OUT/'selected_hyperparameters.csv', index=False)
primary_scheme = selected.sort_values(['validation_mse', 'scheme']).iloc[0].scheme
PARAMS = {row.scheme: dict(max_leaf_nodes=int(row.max_leaf_nodes), max_iter=int(row.max_iter),
                        l2_regularization=float(row.l2_regularization),
                        min_samples_leaf=int(row.min_samples_leaf))
        for row in selected.itertuples()}
config.update(primary_scheme=primary_scheme, selected=selected.to_dict('records'))
(OUT/'model_config.json').write_text(json.dumps(config, indent=2)+'\n', encoding='utf-8')
display(selected)
print('Validation-selected scheme:', primary_scheme)


## 4. Selected-model forecasts and test evaluation

Refit the early-stopping-selected configurations monthly on all eligible historical
rows to export validation and test forecasts as `gbdt`. The chosen leaf count,
minimum leaf size, L2 penalty and number of rounds stay fixed throughout the
test period.
Test feature-group importance uses three joint within-month permutations per group.


In [ ]:
def fit_month(task):
    month, ticker, scheme = task
    x_train, y_train, x_pred, result, audit = month_data(month, ticker, scheme)
    parameters = PARAMS[scheme]
    model = HistGradientBoostingRegressor(**FIXED, **parameters)
    model.fit(x_train, y_train)
    result['y_hat'] = model.predict(x_pred)
    audit.update(model='gbdt', **parameters)

    importance = []
    if month >= '2023-01':
        score = result.common_scoreable.to_numpy()
        x_score = x_pred[score]
        y_score = result.loc[score, 'target_relative_return'].to_numpy()
        base_sse = np.square(y_score-result.loc[score, 'y_hat'].to_numpy()).sum()
        names = spec['predictors_by_target'][ticker][scheme]
        rng = np.random.default_rng(SEED + int(month.replace('-', '')))
        orders = [rng.permutation(len(y_score)) for _ in range(3)]
        for group in sorted({name.split('_')[0] for name in names}):
            indices = [i for i, name in enumerate(names) if name.startswith(group+'_')]
            changes = []
            for order in orders:
                shuffled = x_score.copy()
                shuffled[:, indices] = x_score[order][:, indices]
                prediction = model.predict(shuffled)
                changes.append(np.square(y_score-prediction).sum()-base_sse)
            importance.append(dict(scheme=scheme, feature_group=group, n=len(y_score),
                                sse_increase=np.mean(changes)))
    return result, audit, importance

frames, audits, importance_rows = [], [], []
started = time.perf_counter()
with threadpool_limits(limits=1), Parallel(n_jobs=4, prefer="threads") as pool:
    for month in schedule:
        tasks = [(month, ticker, scheme) for ticker in sorted(data) for scheme in SCHEMES]
        for frame, audit, importance in pool(delayed(fit_month)(task) for task in tasks):
            frames.append(frame)
            audits.append(audit)
            importance_rows.extend(importance)
        print('Selected models', month, '|', len(audits), 'fits |', round(time.perf_counter()-started, 1), 's')

predictions = pd.concat(frames, ignore_index=True).sort_values(['date', 'ticker', 'scheme'])
handoff = ['date', 'ticker', 'model', 'scheme', 'model_month', 'fit_cutoff', 'y_hat']
predictions[handoff].to_csv(OUT/'gbdt_predictions.csv', index=False)
predictions.to_csv(OUT/'gbdt_predictions_full.csv', index=False)
pd.DataFrame(audits).to_csv(OUT/'fit_audit.csv', index=False)
group_importance = pd.DataFrame(importance_rows).groupby(
    ['scheme', 'feature_group'], as_index=False)[['sse_increase', 'n']].sum()
group_importance['mse_increase'] = group_importance.sse_increase/group_importance.n
group_importance.to_csv(OUT/'group_permutation_importance.csv', index=False)
print('Forecasts:', len(predictions))

## 5. Forecast accuracy

Out-of-sample $R^2 = 1-\mathrm{SSE}_{\mathrm{model}}/\mathrm{SSE}_{\mathrm{historical\ mean}}$.
`gbdt` is the prediction from the tree model selected with early stopping.

In [ ]:
en = pd.read_csv(ROOT/'results/elastic_net/elastic_net_predictions_full.csv',
                parse_dates=['date'], float_precision='round_trip')
columns = ['date', 'ticker', 'split', 'scheme', 'model', 'common_scoreable',
        'target_relative_return', 'historical_mean', 'y_hat']
combined = pd.concat([predictions[columns], en[columns]], ignore_index=True)
scored = combined.loc[combined.common_scoreable].copy()

def metric_table(frame, keys):
    rows = []
    for key, g in frame.groupby(keys):
        y, p, mean = g.target_relative_return.to_numpy(), g.y_hat.to_numpy(), g.historical_mean.to_numpy()
        error = y-p
        mse = np.mean(error**2)
        rows.append(dict(zip(keys, key)) | dict(
            n=len(g), MSE=mse, RMSE=np.sqrt(mse), MAE=np.mean(abs(error)),
            R2_OOS_mean=1-np.sum(error**2)/np.sum((y-mean)**2),
            R2_OOS_zero=1-np.sum(error**2)/np.sum(y**2),
            direction_accuracy=np.mean(np.sign(y)==np.sign(p))))
    return pd.DataFrame(rows)

comparison = metric_table(scored, ['split', 'model', 'scheme'])
comparison.to_csv(OUT/'model_comparison_metrics.csv', index=False)
metrics = comparison.loc[comparison.model.eq('gbdt')].drop(columns='model')
metrics.to_csv(OUT/'forecast_metrics.csv', index=False)
gbdt = scored.loc[scored.model.eq('gbdt')].copy()
sector = metric_table(gbdt, ['split', 'scheme', 'ticker'])
sector.to_csv(OUT/'sector_forecast_metrics.csv', index=False)
annual = metric_table(gbdt.assign(year=gbdt.date.dt.year), ['split', 'scheme', 'year'])
annual.to_csv(OUT/'annual_forecast_metrics.csv', index=False)
display(metrics)
display(comparison.loc[comparison.scheme.eq(primary_scheme)])

## 6. Paired loss differences

Average losses across sectors by date and compute approximate 95% Newey–West
intervals with 20 lags. Positive improvement means lower MSE for the model being compared.

In [ ]:
scored['model_loss'] = (scored.target_relative_return-scored.y_hat)**2
scored['mean_loss'] = (scored.target_relative_return-scored.historical_mean)**2
daily = scored.groupby(['split', 'scheme', 'date', 'model'], as_index=False)[
    ['model_loss', 'mean_loss']].mean()

def loss_interval(values):
    x = np.asarray(values)
    z, n = x-x.mean(), len(x)
    variance = z@z/n
    for lag in range(1, 21):
        variance += 2*(1-lag/21)*(z[lag:]@z[:-lag])/n
    se = np.sqrt(variance/n)
    return dict(mean_loss_improvement=x.mean(), lower_95=x.mean()-1.96*se,
                upper_95=x.mean()+1.96*se, days=n)


rows = []
for (split, scheme), g in daily.groupby(['split', 'scheme']):
    losses = g.pivot(index='date', columns='model', values='model_loss')
    losses['historical_mean'] = g.loc[g.model.eq('gbdt')].set_index('date').mean_loss
    for baseline in ['historical_mean', 'elastic_net']:
        rows.append(dict(split=split, scheme=scheme, model='gbdt', baseline=baseline,
                         **loss_interval(losses[baseline]-losses.gbdt)))
uncertainty = pd.DataFrame(rows)
uncertainty.to_csv(OUT/'loss_difference_uncertainty.csv', index=False)
display(uncertainty.loc[uncertainty['split'].eq('test')])


## 7. Results

In [ ]:
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, split in zip(axes, ['validation', 'test']):
    table = comparison.loc[comparison['split'].eq(split)].pivot(
        index='scheme', columns='model', values='R2_OOS_mean')
    table[['elastic_net', 'gbdt']].plot.bar(ax=ax, rot=0)
    for bars in ax.containers:
        ax.bar_label(bars, fmt='%.6f', padding=3, fontsize=8)
    ax.margins(y=0.18)
    ax.axhline(0, color='black', lw=.7)
    ax.set(title=split.title(), xlabel='')
handles, labels = axes[0].get_legend_handles_labels()
for ax in axes:
    ax.get_legend().remove()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)
axes[0].set_ylabel('OOS R² vs historical mean')
fig.tight_layout(rect=(0, 0, 1, .9))
fig.savefig(OUT/'01_model_comparison.png')
plt.show()

table = sector.loc[sector['split'].eq('test')].pivot(
    index='ticker', columns='scheme', values='R2_OOS_mean')*100
fig, ax = plt.subplots(figsize=(7, 5))
limit = np.abs(table.to_numpy()).max()
im = ax.imshow(table, cmap='RdBu', vmin=-limit, vmax=limit, aspect='auto')
ax.set_xticks(range(len(table.columns)), table.columns)
ax.set_yticks(range(len(table.index)), table.index)
for i in range(len(table.index)):
    for j in range(len(table.columns)):
        ax.text(j, i, f'{table.iloc[i, j]:.2f}', ha='center', va='center',
                color='white' if abs(table.iloc[i, j])>.65*limit else 'black')
ax.set_title('Test OOS R² by sector (%)')
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(OUT/'02_sector_r2.png')
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
(annual.pivot(index='year', columns='scheme', values='R2_OOS_mean')*100).plot(ax=ax, marker='o')
ax.axhline(0, color='black', lw=.7)
ax.axvline(2022.5, color='gray', ls='--')
ax.set(title='Annual GBDT performance', ylabel='OOS R² (%)', xlabel='Year')
fig.tight_layout()
fig.savefig(OUT/'03_annual_r2.png')
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
group_importance.pivot(index='feature_group', columns='scheme', values='mse_increase').plot.bar(ax=ax, rot=0)
ax.axhline(0, color='black', lw=.7)
ax.set(title='Test group permutation importance', ylabel='MSE increase after shuffling', xlabel='')
fig.tight_layout()
fig.savefig(OUT/'04_group_importance.png')
plt.show()


`validation_round_curve.csv` records pooled validation loss at each round.
`validation_grid.csv` records the best visited round and patience stopping point
for each leaf-count/minimum-leaf-size/L2 configuration.
`selected_hyperparameters.csv` records the chosen configuration per scheme.

`gbdt_predictions.csv` is the seven-column forecast handoff in decimal-return
units. `fit_audit.csv` records the training sample and selected parameters,
including minimum leaf size, for each month. `model_comparison_metrics.csv` compares GBDT and Elastic Net.
